# Exploratory Data Analysis
---

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

FILE_PATH  = '../../data/bronze/transactions_train.csv'
CHUNK_SIZE = 100_000          # rows to read at a time (keeps RAM low)
SAMPLE_N   = 50_000           # rows to use for visualization

## 1. Basic Info

In [2]:
import os

# Check file size without loading it
size_gb = os.path.getsize(FILE_PATH) / 1e9
print(f'File size: {size_gb:.2f} GB')

# Read only first 5 rows to see columns
preview = pd.read_csv(FILE_PATH, nrows=5)
preview

File size: 3.49 GB


,t_dat,customer_id,article_id,price,sales_channel_id
0,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,663713001,0.050831,2
1,2018-09-20,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,541518023,0.030492,2
2,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,505221004,0.015237,2
3,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687003,0.016932,2
4,2018-09-20,00007d2de826758b65a93dd24ce629ed66842531df6699...,685687004,0.016932,2


## 2. Row Count & Missing Values
We read the file in chunks so it never fully loads into RAM.

In [3]:
total_rows = 0
null_counts = None

for chunk in pd.read_csv(FILE_PATH, chunksize=CHUNK_SIZE):
    total_rows += len(chunk)
    if null_counts is None:
        null_counts = chunk.isnull().sum()
    else:
        null_counts += chunk.isnull().sum()

print(f'Total rows: {total_rows:,}')
print(f'Total columns: {len(null_counts)}')

Total rows: 31,788,324
Total columns: 5


In [4]:
# Missing value summary table
missing = pd.DataFrame({
    'Missing Count': null_counts,
    'Missing %'    : (null_counts / total_rows * 100).round(1)
}).sort_values('Missing %', ascending=False)

missing[missing['Missing Count'] > 0]

,Missing Count,Missing %


## 3. Take a Sample for Visualization
50k rows is more than enough to see patterns.

In [ ]:
# skiprows randomly skips lines — simple and memory-friendly
import numpy as np

skip = sorted(
    np.random.choice(range(1, total_rows + 1),
                     size=total_rows - SAMPLE_N,
                     replace=False)
)

df = pd.read_csv(FILE_PATH, skiprows=skip)
df.columns = preview.columns  # restore column names

print(f'Sample size: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

## 4. Data Types

In [ ]:
# Shows each column's data type
df.dtypes.to_frame('Type')

## 5. Descriptive Statistics

In [ ]:
# Basic stats for numeric columns
pd.options.display.float_format = '{:.2f}'.format
df.describe()

## 6. Missing Value Chart

In [ ]:
miss_pct = (null_counts / total_rows * 100).sort_values(ascending=False)
miss_pct = miss_pct[miss_pct > 0]  # only columns with missing values

if miss_pct.empty:
    print('No missing values!')
else:
    fig, ax = plt.subplots(figsize=(10, 4))
    bars = ax.bar(miss_pct.index, miss_pct.values, color='#e74c3c', alpha=0.8)
    ax.set_ylabel('Missing %')
    ax.set_title('Missing Values by Column')
    ax.set_xticklabels(miss_pct.index, rotation=45, ha='right')
    ax.axhline(20, color='gray', linestyle='--', alpha=0.5, label='20% threshold')
    ax.legend()
    plt.tight_layout()
    plt.show()

## 7. Distributions of Numeric Columns

In [ ]:
num_cols = df.select_dtypes(include='number').columns.tolist()
n = len(num_cols)

if n == 0:
    print('No numeric columns found.')
else:
    ncols = 3
    nrows = (n + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 3))
    axes = axes.flatten()

    for ax, col in zip(axes, num_cols):
        df[col].dropna().hist(ax=ax, bins=30, color='#3498db', edgecolor='white')
        ax.set_title(col, fontweight='bold')
        ax.set_xlabel('')

    for ax in axes[n:]:
        ax.set_visible(False)

    plt.suptitle('Numeric Column Distributions', y=1.01, fontsize=13)
    plt.tight_layout()
    plt.show()

## 8. Categorical Columns — Top Values

In [ ]:
# Keep only columns with 2–30 unique values (likely categorical)
cat_cols = [c for c in df.columns if 2 <= df[c].nunique() <= 30]

summary = pd.DataFrame({
    "unique": [df[c].nunique() for c in cat_cols],
    "dtype" : [df[c].dtype       for c in cat_cols]
}, index=cat_cols)

print(f"Found {len(cat_cols)}/{len(df.columns)} categorical columns\n")
display(summary)

## 9. Correlation Heatmap

In [ ]:
if len(num_cols) < 2:
    print('Need at least 2 numeric columns for correlation.')
else:
    corr = df[num_cols].corr()
    size = max(6, len(num_cols))

    fig, ax = plt.subplots(figsize=(size, size - 1))
    im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
    plt.colorbar(im, ax=ax)

    ax.set_xticks(range(len(num_cols)))
    ax.set_yticks(range(len(num_cols)))
    ax.set_xticklabels(num_cols, rotation=45, ha='right')
    ax.set_yticklabels(num_cols)

    # Write correlation value inside each cell
    for i in range(len(num_cols)):
        for j in range(len(num_cols)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}",
                    ha='center', va='center', fontsize=8)

    ax.set_title('Correlation Matrix', fontsize=13)
    plt.tight_layout()
    plt.show()

## 10. Quick Summary

In [ ]:
print('DATASET SUMMARY')
print('-' * 35)
print(f'Total rows    : {total_rows:,}')
print(f'Columns       : {len(null_counts)}')
print(f'Numeric cols  : {len(num_cols)}')
print(f'Categorical   : {len(cat_cols)}')
print()

high_miss = missing[missing['Missing %'] > 20]
if len(high_miss) > 0:
    print(f'Warning: {len(high_miss)} column(s) have more than 20% missing values.')
    print(high_miss.to_string())
else:
    print('No major missing value issues.')